# Final Self-Healing RAG deployment — BERT runtime 4.2

Same workflow as the original project: **Google Drive → Colab T4 → tunnel → web dashboard**.
Extract the ZIP and upload its `pjt` folder to `/MyDrive/pjt`. Start a **fresh T4 runtime**, then run these cells in order.
The accepted BERT weights are included. This notebook does **not** retrain the classifier or repeat the evaluation.
An existing matching index is reused; otherwise the first run builds the same 5,000-record Wikipedia corpus plus the 40 supplied source paragraphs and saves it to Drive.


## 1. Mount Drive, copy project and install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/pjt')
LOCAL_DIR = Path('/content/pjt')
if not (DRIVE_DIR / 'data/bert_failure_classifier_runtime_v4/model.safetensors').is_file():
    raise FileNotFoundError('Extract the supplied ZIP first, then upload its pjt folder to MyDrive/pjt, including data/.')
shutil.copytree(DRIVE_DIR, LOCAL_DIR, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('node_modules', '__pycache__', '.git', '.venv'))
os.chdir(LOCAL_DIR)
if str(LOCAL_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Working project:', LOCAL_DIR)


## 2. Prepare or reuse the final search index

In [ ]:
from src.prepare_deployment import prepare, check_assets
prepare()

# Persist both indices so later Colab sessions reuse them.
for name in ['faiss_index', 'faiss_index_with_source_passages']:
    source = LOCAL_DIR / 'data' / name
    if source.is_dir():
        shutil.copytree(source, DRIVE_DIR / 'data' / name, dirs_exist_ok=True)
print(check_assets())
print('Index cache saved in your project data folder on Drive.')


## 3. Load the final models

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
from src import api_server
api_server.initialize_runtime()
print('Final BERT, Mistral, MiniLM, NLI and reranker loaded.')
print('Search vectors:', api_server.pipeline_instance.index.ntotal)
print('No classifier training or benchmark evaluation was run.')


## 4. Start the REST API and dashboard

In [ ]:
import threading, time, urllib.request, json
import uvicorn
if '_rag_server' not in globals() or not _rag_server.started:
    _rag_server = uvicorn.Server(uvicorn.Config(api_server.app, host='0.0.0.0', port=8000,
                                              log_level='info', access_log=False))
    _rag_server_thread = threading.Thread(target=_rag_server.run, daemon=True)
    _rag_server_thread.start()
for _ in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3) as response:
            health = json.load(response)
        assert health['ready'] and health['version'] == '4.2'
        print('API ready:', health)
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError('API did not become ready. Check the model-loading cell and server log.')


## 5. Open the dashboard through Cloudflare

In [ ]:
import re
from IPython.display import display, HTML
binary = Path('/content/cloudflared')
if not binary.is_file():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        binary)
    binary.chmod(0o755)
if '_rag_tunnel' in globals() and _rag_tunnel.poll() is None:
    print('Existing tunnel is still running:', globals().get('PUBLIC_URL', 'see tunnel log'))
else:
    log_path = Path('/content/rag-tunnel.log')
    _rag_tunnel_log = log_path.open('w')
    _rag_tunnel = subprocess.Popen([str(binary), 'tunnel', '--url', 'http://127.0.0.1:8000',
                                   '--no-autoupdate', '--protocol', 'http2'],
                                  stdout=_rag_tunnel_log, stderr=subprocess.STDOUT)
    for _ in range(180):
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_path.read_text(errors='replace'))
        if match:
            PUBLIC_URL = match.group(0)
            display(HTML(f'<p><a href="{PUBLIC_URL}" target="_blank" rel="noopener">Open Self-Healing RAG dashboard</a></p>'))
            print('Frontend and backend:', PUBLIC_URL)
            print('If using a separately hosted frontend, paste this URL into Gateway settings.')
            break
        if _rag_tunnel.poll() is not None:
            raise RuntimeError(log_path.read_text(errors='replace'))
        time.sleep(0.5)
    else:
        raise RuntimeError('No tunnel URL yet. Check /content/rag-tunnel.log and retry this cell.')


Keep the Colab runtime connected while using the dashboard. The temporary tunnel URL changes when you restart it.

To stop manually, run `_rag_tunnel.terminate()` and set `_rag_server.should_exit = True` in a new cell.

For the optional original Gradio view, install `requirements-gradio.txt` before using `create_demo_app` from `src.ui_app`. The HTML dashboard above needs no Gradio installation and exposes the full before/after signals, BERT probabilities, evidence and recovery trace.
